# Tarea 1 — Analisis Completo de Datos
## Marketing Analytics para Empresa Fintech

**Objetivo**: Analizar los datos de campanas de marketing directo de un banco portugues para identificar patrones, tendencias y factores que influyen en la contratacion de depositos a plazo.

**Dataset**: `bank-additional-full.csv` — 41.188 registros, 21 variables

**Modo**: Solo lectura — el CSV original no se modifica.

### Estructura del Notebook
1. Carga y esquema del dataset
2. Calidad de datos
3. Analisis de valores unknown
4. Propuesta de renombrado semantico
5. Estadisticas descriptivas (numericas)
6. Estadisticas descriptivas (categoricas)
7. Deteccion de outliers
8. Variable target y desbalance
9. Correlaciones
10. Multicolinealidad
11. Relacion demografica con la suscripcion
12. Impacto de la campana (contactos, timing)
13. Duracion de la llamada
14. Historial de campanas anteriores
15. Variables macroeconomicas y conclusiones

## 1. Carga del Dataset y Esquema

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, spearmanr
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Carga del dataset — SOLO LECTURA, no se modifica el CSV
CSV = "../../datos/bank-additional_bank-additional-full.csv"
df = pd.read_csv(CSV, sep=";")
n, p = df.shape
print(f"Dataset cargado: {n:,} filas x {p} columnas")
print(f"\nColumnas: {list(df.columns)}")
df.head()

In [ ]:
# Clasificacion de variables
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Variables numericas ({len(num_cols)}): {num_cols}")
print(f"Variables categoricas ({len(cat_cols)}): {cat_cols}")

schema = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "tipo_analisis": ["numerica" if c in num_cols else "categorica" for c in df.columns],
    "n_unicos": df.nunique(),
    "ejemplo": [str(df[c].iloc[0]) for c in df.columns]
})
schema

## 2. Calidad de Datos

In [ ]:
# Valores nulos
null_check = df.isnull().sum()
print(f"Total valores NaN: {null_check.sum()}")

# Duplicados
dupes = df.duplicated().sum()
print(f"Filas duplicadas: {dupes} de {n:,} ({dupes/n*100:.3f}%)")

# Codigos especiales
pdays_999 = (df["pdays"] == 999).sum()
print(f"\npdays = 999 (nunca contactado): {pdays_999:,} ({pdays_999/n*100:.1f}%)")

dur_0 = (df["duration"] == 0).sum()
print(f"duration = 0 (llamada no conectada): {dur_0}")

## 3. Analisis de Valores Unknown

Cuatro variables contienen `"unknown"`: default, education, job, marital. En lugar de eliminarlos o reemplazar por NaN, se analizan como posibles perfiles diferenciados.

In [ ]:
# Conteo de unknowns por columna
COLS_UNK = ["job", "marital", "education", "default", "housing", "loan"]

print("CONTEO DE UNKNOWNS POR COLUMNA:")
print("-" * 60)
for col in COLS_UNK:
    cnt = (df[col] == "unknown").sum()
    if cnt > 0:
        print(f"  {col:20s}: {cnt:>6,} ({cnt/n*100:.2f}%)")

# Co-ocurrencia
df_unk = pd.DataFrame()
for col in COLS_UNK:
    df_unk[f"{col}_unk"] = (df[col] == "unknown").astype(int)

df_unk["total_unknowns"] = df_unk.sum(axis=1)

print("\nRegistros por cantidad de columnas unknown:")
vc = df_unk["total_unknowns"].value_counts().sort_index()
for k, v in vc.items():
    print(f"  {k} unknowns: {v:>6,} registros ({v/n*100:.2f}%)")

In [ ]:
# Tasa de conversion: unknown vs known
df["y_num"] = (df["y"] == "yes").astype(int)
overall_conv = df["y_num"].mean() * 100

print("TASA DE CONVERSION: unknown vs known")
print("-" * 70)
for col in ["default", "education", "job", "marital"]:
    is_unk = df[col] == "unknown"
    if is_unk.sum() == 0:
        continue
    rate_unk = df.loc[is_unk, "y_num"].mean() * 100
    rate_known = df.loc[~is_unk, "y_num"].mean() * 100
    cnt_unk = is_unk.sum()
    diff = rate_unk - rate_known
    print(f"  {col:20s}: known={rate_known:.2f}%  unknown={rate_unk:.2f}%  (diff={diff:+.2f}pp, n={cnt_unk:,})")

# Chi-cuadrado: Es 'unknown' informativo?
print("\nCHI-CUADRADO: Es 'unknown' informativo para la target?")
print("-" * 60)
for col in ["default", "education", "job", "marital"]:
    has_unk = (df[col] == "unknown").sum()
    if has_unk == 0:
        continue
    ct = pd.crosstab(df[col] == "unknown", df["y"])
    chi2, pval, dof, _ = chi2_contingency(ct)
    sig = "SIGNIFICATIVO" if pval < 0.05 else "no significativo"
    print(f"  {col:20s}: chi2={chi2:.2f}, p={pval:.6f} -> {sig}")

## 4. Propuesta de Renombrado Semantico

Tras analizar el perfil de cada grupo de unknowns, se proponen nombres que reflejan su comportamiento real:

| Variable | unknown -> | Justificacion |
|:---------|:-----------|:------------|
| **default** | `no_credit_record` | Perfil conservador sin historial crediticio. Conv. 5.15% (mitad que 'no') |
| **education** | `undisclosed_education` | Convierten 14.5% (por encima de la media). No son ruido |
| **job** | `undeclared_occupation` | 0.8% del dataset. Conv. 11.2% (en la media) |
| **marital** | `undisclosed_status` | Solo 80 casos. Conv. 15.0% (la mas alta) |

In [ ]:
# Perfilado profundo: default=unknown vs default=no
print("PERFIL: default=unknown vs default=no")
print("-" * 60)
for val in ['no', 'unknown']:
    subset = df[df['default'] == val]
    print(f"\ndefault='{val}' (n={len(subset):,}):")
    print(f"  Edad media:      {subset['age'].mean():.1f}")
    print(f"  Duration media:  {subset['duration'].mean():.1f}s")
    print(f"  Campaign media:  {subset['campaign'].mean():.1f}")
    print(f"  Conversion:      {(subset['y']=='yes').mean()*100:.2f}%")
    never = (subset['pdays']==999).mean()*100
    print(f"  Nunca contactado: {never:.1f}%")
    top_jobs = subset['job'].value_counts(normalize=True).head(3)
    print(f"  Top 3 jobs:      {dict(top_jobs.round(3))}")

In [ ]:
# Aplicar nombres semanticos (solo en memoria)
SEMANTIC_MAP = {
    "default": {"unknown": "no_credit_record"},
    "education": {"unknown": "undisclosed_education"},
    "job": {"unknown": "undeclared_occupation"},
    "marital": {"unknown": "undisclosed_status"},
}
for col, mapping in SEMANTIC_MAP.items():
    df[col] = df[col].replace(mapping)

print("Nombres semanticos aplicados (solo en memoria, CSV intacto).")
print(f"Conversion global: {overall_conv:.2f}%")

## 5. Estadisticas Descriptivas — Variables Numericas

In [ ]:
# Descriptivas completas
desc = df[num_cols].describe().T
desc["cv%"] = (desc["std"] / desc["mean"].abs() * 100).round(1)
desc["mediana"] = df[num_cols].median()
desc["asimetria"] = df[num_cols].skew().round(3)
desc["kurtosis"] = df[num_cols].kurtosis().round(3)
desc[["count", "mean", "std", "min", "25%", "mediana", "75%", "max", "cv%", "asimetria", "kurtosis"]]

## 6. Estadisticas Descriptivas — Variables Categoricas

In [ ]:
# Distribucion de cada variable categorica
for col in cat_cols:
    if col == "y":
        continue
    print(f"\n{'='*50}")
    print(f"  {col.upper()} ({df[col].nunique()} categorias)")
    print(f"{'='*50}")
    vc = df[col].value_counts()
    for val, cnt in vc.items():
        conv = df.loc[df[col]==val, "y_num"].mean()*100
        print(f"  {val:<25s}: {cnt:>6,} ({cnt/n*100:>5.1f}%) | conv={conv:.2f}%")

## 7. Deteccion de Outliers (Metodo IQR x 1.5)

In [ ]:
print(f"{'Variable':<20s} {'Q1':>10s} {'Q3':>10s} {'IQR':>10s} {'Lim.Inf':>10s} {'Lim.Sup':>10s} {'%Outliers':>10s}")
print("-" * 75)
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = outliers / n * 100
    if pct > 0:
        print(f"  {col:<20s} {q1:>10.2f} {q3:>10.2f} {iqr:>10.2f} {lower:>10.2f} {upper:>10.2f} {pct:>9.2f}%")

## 8. Variable Target — Desbalance de Clases

In [ ]:
# Distribucion de y
yes = (df["y"] == "yes").sum()
no = (df["y"] == "no").sum()
ratio = no / yes

print(f"  no:  {no:>6,} ({no/n*100:.2f}%)")
print(f"  yes: {yes:>6,} ({yes/n*100:.2f}%)")
print(f"\n  Ratio no:yes = {ratio:.2f}:1")
print(f"\n  DESBALANCE SIGNIFICATIVO: Solo 1 de cada {ratio:.0f} clientes contrata.")
print(f"  Para modelado: usar SMOTE, class_weight='balanced', o ajustar threshold.")

## 9. Correlaciones con la Variable Target

In [ ]:
# Correlaciones con y_num
corr_target = df[num_cols].corrwith(df["y_num"]).sort_values(ascending=False)
print("Correlaciones con la variable target (y):")
print("-" * 50)
for var, r in corr_target.items():
    direction = "+" if r > 0 else "-"
    bar = "|" * int(abs(r) * 50)
    print(f"  {var:<20s}: {r:>+.4f}  {direction}{bar}")

## 10. Multicolinealidad entre Variables

In [ ]:
# Pares con correlacion alta (|r| > 0.70)
corr_matrix = df[num_cols].corr()
print("PARES CON ALTA MULTICOLINEALIDAD (|r| > 0.70):")
print("-" * 70)
seen = set()
for i in range(len(num_cols)):
    for j in range(i+1, len(num_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.70:
            pair = f"{num_cols[i]:20s} x {num_cols[j]:20s}"
            if pair not in seen:
                print(f"  {pair}: r={r:.4f}")
                seen.add(pair)

print("\n  ALERTA: emp.var.rate, euribor3m y nr.employed son casi la misma variable (r>0.90)")
print("  Para modelado: usar solo 1 (preferiblemente euribor3m) o aplicar PCA.")

## 11. Relacion Demografica con la Suscripcion

Analisis de como edad, trabajo, educacion, estado civil, default, hipoteca y prestamo se relacionan con la contratacion del deposito.

In [ ]:
# EDAD
bins = [0, 30, 40, 50, 60, 100]
labels = ["<30", "30-40", "41-50", "51-60", "60+"]
df["age_band"] = pd.cut(df["age"], bins=bins, labels=labels)

print("EDAD vs SUSCRIPCION:")
print(f"  {'Banda':<10s} {'n':>8s} {'yes':>6s} {'Conv%':>8s} {'vs.Media':>10s}")
print("  " + "-" * 46)
for band in labels:
    subset = df[df["age_band"] == band]
    conv = subset["y_num"].mean() * 100
    diff = conv - overall_conv
    print(f"  {band:<10s} {len(subset):>8,} {subset['y_num'].sum():>6,} {conv:>7.2f}% {diff:>+9.2f}pp")

# Test Mann-Whitney
yes_age = df.loc[df["y"]=="yes", "age"]
no_age = df.loc[df["y"]=="no", "age"]
u, p_age = mannwhitneyu(yes_age, no_age)
print(f"\n  Mann-Whitney U: p={p_age:.4e} -> {'SIGNIFICATIVA' if p_age < 0.05 else 'no sig.'}")
print(f"  Media yes={yes_age.mean():.1f} vs no={no_age.mean():.1f}")

In [ ]:
# JOB
print("\nTIPO DE TRABAJO vs SUSCRIPCION:")
job_conv = df.groupby("job")["y_num"].agg(["count","sum","mean"])
job_conv.columns = ["n","yes","conv"]
job_conv["conv"] = job_conv["conv"] * 100
job_conv = job_conv.sort_values("conv", ascending=False)

print(f"  {'Job':<25s} {'n':>8s} {'yes':>6s} {'Conv%':>8s} {'vs.Media':>10s}")
print("  " + "-" * 60)
for job, row in job_conv.iterrows():
    diff = row["conv"] - overall_conv
    marker = " *" if abs(diff) > 5 else ""
    print(f"  {job:<25s} {row['n']:>8,.0f} {row['yes']:>6,.0f} {row['conv']:>7.2f}% {diff:>+9.2f}pp{marker}")

chi2, p, _, _ = chi2_contingency(pd.crosstab(df["job"], df["y"]))
print(f"\n  Chi2 (job x y): p={p:.4e} -> {'SIGNIFICATIVA' if p < 0.05 else 'no sig.'}")

In [ ]:
# EDUCACION
print("NIVEL EDUCATIVO vs SUSCRIPCION:")
edu_conv = df.groupby("education")["y_num"].agg(["count","sum","mean"])
edu_conv.columns = ["n","yes","conv"]
edu_conv["conv"] = edu_conv["conv"] * 100
edu_conv = edu_conv.sort_values("conv", ascending=False)

print(f"  {'Educacion':<30s} {'n':>8s} {'yes':>6s} {'Conv%':>8s}")
print("  " + "-" * 55)
for edu, row in edu_conv.iterrows():
    print(f"  {edu:<30s} {row['n']:>8,.0f} {row['yes']:>6,.0f} {row['conv']:>7.2f}%")

chi2, p, _, _ = chi2_contingency(pd.crosstab(df["education"], df["y"]))
print(f"\n  Chi2 (education x y): p={p:.4e}")

In [ ]:
# ESTADO CIVIL, DEFAULT, HOUSING, LOAN
for col_name, label in [("marital","Estado Civil"),("default","Default"),("housing","Hipoteca"),("loan","Prestamo")]:
    print(f"\n{label.upper()} vs SUSCRIPCION:")
    col_conv = df.groupby(col_name)["y_num"].agg(["count","sum","mean"])
    col_conv.columns = ["n","yes","conv"]
    col_conv["conv"] = col_conv["conv"] * 100
    for val, row in col_conv.iterrows():
        print(f"  {val:<25s}: {row['conv']:>6.2f}% (n={row['n']:,.0f})")
    chi2, p, _, _ = chi2_contingency(pd.crosstab(df[col_name], df["y"]))
    print(f"  Chi2: p={p:.4e} -> {'SIGNIFICATIVA' if p < 0.05 else 'no sig.'}")

**Hallazgos demograficos clave:**
- **Edad**: Patron en "U" — jovenes (<30) y mayores (60+) convierten mas
- **Job**: student (31.5%) y retired (25.2%) son los top converters
- **Default**: `no_credit_record` (5.15%) convierte la MITAD que `no` (12.88%). Es el 20.9% del dataset
- **Todas** las variables demograficas son significativas (p < 0.001)

## 12. Impacto de la Campana — Contactos, Mes y Dia

In [ ]:
# NUMERO DE CONTACTOS Y SATURACION
print("NUMERO DE CONTACTOS (campaign) vs SUSCRIPCION:")
print(f"  {'Contactos':>10s} {'n':>8s} {'yes':>6s} {'Conv%':>8s}")
print("  " + "-" * 36)

for c in range(1, 16):
    if c < 15:
        subset = df[df["campaign"] == c]
        label = str(c)
    else:
        subset = df[df["campaign"] >= 15]
        label = "15+"
    if len(subset) > 0:
        conv = subset["y_num"].mean() * 100
        print(f"  {label:>10s} {len(subset):>8,} {subset['y_num'].sum():>6,} {conv:>7.2f}%")

# Punto de saturacion
print("\nANALISIS DE SATURACION:")
for threshold in [1, 2, 3, 5]:
    above = df[df["campaign"] > threshold]
    below = df[df["campaign"] <= threshold]
    conv_b = below["y_num"].mean() * 100
    conv_a = above["y_num"].mean() * 100 if len(above) > 0 else 0
    print(f"  <=({threshold}) contactos: {conv_b:.2f}% | >{threshold}: {conv_a:.2f}% | Delta={conv_a-conv_b:+.2f}pp")

rho, p_sp = spearmanr(df["campaign"], df["y_num"])
print(f"\n  Spearman: rho={rho:.4f}, p={p_sp:.4e} -> Mas contactos = MENOR conversion")

In [ ]:
# MES DEL CONTACTO
month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]

print("\nMES DEL ULTIMO CONTACTO vs SUSCRIPCION:")
print(f"  {'Mes':<6s} {'n':>8s} {'%Vol':>7s} {'yes':>6s} {'Conv%':>8s}")
print("  " + "-" * 40)
for m in month_order:
    subset = df[df["month"] == m]
    if len(subset) > 0:
        conv = subset["y_num"].mean() * 100
        vol = len(subset) / n * 100
        print(f"  {m:<6s} {len(subset):>8,} {vol:>6.1f}% {subset['y_num'].sum():>6,} {conv:>7.2f}%")

# DIA DE LA SEMANA
print("\nDIA DE LA SEMANA:")
for d in ["mon","tue","wed","thu","fri"]:
    subset = df[df["day_of_week"] == d]
    conv = subset["y_num"].mean() * 100
    print(f"  {d}: {conv:.2f}%")

## 13. Duracion de la Llamada

> **Nota**: `duration` no se conoce antes de realizar la llamada. No es predictor a priori, pero indica el interes del cliente.

In [ ]:
# Buckets de duracion con intervalos [a, b) para evitar ambiguedad
buckets = [
    (0,    60,    "< 1 min"),
    (60,   180,   "1-3 min"),
    (180,  300,   "3-5 min"),
    (300,  600,   "5-10 min"),
    (600,  1200,  "10-20 min"),
    (1200, 99999, "20+ min"),
]

print("DURACION vs SUSCRIPCION:")
print(f"  {'Bucket':<15s} {'Rango':>15s} {'n':>8s} {'Conv%':>8s}")
print("  " + "-" * 50)
for lo, hi, label in buckets:
    subset = df[(df["duration"] >= lo) & (df["duration"] < hi)]
    conv = subset["y_num"].mean() * 100 if len(subset) > 0 else 0
    print(f"  {label:<15s} {'['+str(lo)+'s,'+str(hi)+'s)':>15s} {len(subset):>8,} {conv:>7.2f}%")

rho, p = spearmanr(df["duration"], df["y_num"])
print(f"\n  Spearman: rho={rho:.4f}, p={p:.4e} -> Mas duracion = MAS suscripcion")

## 14. Historial de Campanas Anteriores y Metodo de Contacto

In [ ]:
# POUTCOME
print("RESULTADO DE CAMPANA ANTERIOR (poutcome):")
for outcome in ["success", "failure", "nonexistent"]:
    subset = df[df["poutcome"] == outcome]
    conv = subset["y_num"].mean() * 100
    print(f"  {outcome:<15s}: {conv:>6.2f}% (n={len(subset):,})")

chi2, p, _, _ = chi2_contingency(pd.crosstab(df["poutcome"], df["y"]))
print(f"  Chi2: p={p:.4e}")

# PDAYS
print("\nDIAS DESDE ULTIMO CONTACTO PREVIO (pdays):")
pdays_999 = df[df["pdays"] == 999]
pdays_not = df[df["pdays"] != 999]
print(f"  pdays=999 (nunca): {pdays_999['y_num'].mean()*100:.2f}% (n={len(pdays_999):,})")
print(f"  pdays!=999 (antes): {pdays_not['y_num'].mean()*100:.2f}% (n={len(pdays_not):,})")

# METODO DE CONTACTO
print("\nMETODO DE CONTACTO:")
for method in ["cellular", "telephone"]:
    subset = df[df["contact"] == method]
    conv = subset["y_num"].mean() * 100
    print(f"  {method:<12s}: {conv:.2f}% (n={len(subset):,})")

## 15. Variables Macroeconomicas y Conclusiones

In [ ]:
# VARIABLES MACROECONOMICAS
macro_vars = [
    ("emp.var.rate",   "Tasa variacion empleo"),
    ("cons.price.idx", "Indice precios consumo"),
    ("cons.conf.idx",  "Indice confianza consumidor"),
    ("euribor3m",      "Euribor 3 meses"),
    ("nr.employed",    "Total empleados (miles)"),
]

print("VARIABLES MACROECONOMICAS vs SUSCRIPCION:")
print(f"  {'Variable':<20s} {'Corr(y)':>8s} {'Media(yes)':>12s} {'Media(no)':>12s} {'p-value':>12s}")
print("  " + "-" * 68)

for var, desc in macro_vars:
    yes_v = df.loc[df["y"]=="yes", var]
    no_v = df.loc[df["y"]=="no", var]
    corr = df[var].corr(df["y_num"])
    u, p = mannwhitneyu(yes_v, no_v)
    print(f"  {var:<20s} {corr:>+8.4f} {yes_v.mean():>12.4f} {no_v.mean():>12.4f} {p:>12.2e}")

print("\n  CONCLUSION: Los depositos se contratan MAS en periodos de incertidumbre")
print("  economica (euribor bajo, empleo en descenso) = refugio seguro para el ahorro.")

In [ ]:
# RESUMEN DE TODOS LOS TESTS ESTADISTICOS
print("\nRESUMEN DE TESTS ESTADISTICOS:")
print(f"  {'Variable':<25s} {'Test':>15s} {'p-value':>15s} {'Resultado':>15s}")
print("  " + "-" * 73)

# Numericas -> Mann-Whitney
for var in ["age", "duration", "campaign"] + [v[0] for v in macro_vars]:
    yes_v = df.loc[df["y"]=="yes", var]
    no_v = df.loc[df["y"]=="no", var]
    u, p = mannwhitneyu(yes_v, no_v)
    sig = "SIGNIFICATIVO" if p < 0.05 else "no sig."
    print(f"  {var:<25s} {'Mann-Whitney':>15s} {p:>15.2e} {sig:>15s}")

# Categoricas -> Chi2
for var in ["job","education","marital","default","housing","loan","contact","month","day_of_week","poutcome"]:
    ct = pd.crosstab(df[var], df["y"])
    chi2, p, _, _ = chi2_contingency(ct)
    sig = "SIGNIFICATIVO" if p < 0.05 else "no sig."
    print(f"  {var:<25s} {'Chi2':>15s} {p:>15.2e} {sig:>15s}")

## Conclusiones Finales — Tarea 1

### Top 10 Factores por Impacto en la Conversion

| # | Factor | Conversion | Tipo |
|:-:|:-------|:----------:|:-----|
| 1 | poutcome = success | 65.11% | Campana |
| 2 | pdays != 999 (contactado antes) | 63.83% | Campana |
| 3 | duration > 20 min | 62.33% | Campana (no predictor) |
| 4 | Euribor bajo + empleo bajo | Significativo | Macroeconomico |
| 5 | job = student | 31.54% | Demografico |
| 6 | job = retired | 25.23% | Demografico |
| 7 | age > 60 | 19.41% | Demografico |
| 8 | contact = cellular | 14.74% | Campana |
| 9 | education = undisclosed | 14.50% | Demografico |
| 10 | campaign <= 3 contactos | 12.17% | Campana |

### Alertas para el Modelado Predictivo (Tarea 3)
- **Desbalance 7.88:1** -> Usar SMOTE o class_weight
- **Multicolinealidad** -> emp.var.rate, euribor3m, nr.employed (r>0.90): usar solo 1
- **duration** -> NO incluir como predictor (data leakage)
- **Unknowns** -> Usar nombres semanticos como categorias propias
- **poutcome** -> Mejor predictor, considerar feature `has_previous_success`

---
**Datos no modificados. CSV original intacto.**

In [ ]:
# Limpieza de columnas temporales
df.drop(columns=["age_band", "y_num"], inplace=True)
print("Notebook completado. Tarea 1 cerrada al 100%.")